# 라벨링 골드셋 from_topic 부여 (개념명 기준)

02-12, 02-13 라벨링 골드셋(v2 파이프라인)에 `from_topic`을 부여한다.

- **개념**: `from_topic` = 개념명 자체 (군집화 없음, 개념 1개 = 토픽 1개)
- **예시/실습**: LLM이 개념 목록 중 내용상 가장 적합한 개념명을 `from_topic`으로 배정
  (위치 무관 → 멀리 있어도 맞는 개념에 붙음)

→ sequence_goldset / error_goldset에서 같은 `from_topic`끼리 묶임

In [13]:
import json
import re
from pathlib import Path

import pandas as pd
import openpyxl
from openpyxl.styles import Alignment
import google.generativeai as genai

import sys
sys.path.insert(0, str(Path(".").resolve().parent))
from app.core.config import settings

genai.configure(api_key=settings.api_key)
MODEL = settings.eval_model

DATES = ["2026-02-10", "2026-02-11", "2026-02-12", "2026-02-13"]  # 02-09 제외
GS_LABEL = Path("../data/goldset/labeling")

KIND = {"concept": "concepts", "example": "examples", "practice": "practices"}
NAME = {"concept": "concept_name", "example": "example_name", "practice": "practice_name"}
print(f"대상: {', '.join(DATES)}  |  model: {MODEL}")

대상: 2026-02-10, 2026-02-11, 2026-02-12, 2026-02-13  |  model: models/gemini-2.5-pro


In [14]:
# ── LLM: 예시/실습의 토픽 추출 ──────────────────────────────────────
def _parse_json(raw):
    text = re.sub(r"^```\w*\s*", "", raw.strip())
    text = re.sub(r"\s*```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        return json.loads(m.group()) if m else {}


EXTRACT_SYS = """당신은 강의 콘텐츠를 분류하는 전문가입니다. 응답은 JSON만 출력하세요."""

# 예시: 해당 예시가 다루는 주제
EXAMPLE_USER = """각 예시 항목이 다루는 주제(토픽)를 추출하세요.

규칙:
- 아래 '참고 개념 목록'에 일치하는 개념이 있으면 그 이름을 그대로 사용하세요.
- 없으면 내용에 맞는 토픽명을 직접 작성하세요.
- 토픽명은 짧고 명확하게.

참고 개념 목록 (강제 아님):
{concepts}

예시 목록 ([인덱스] 이름):
{items}

아래 JSON 형식으로만 응답하세요:
{{"topics": [{{"index": 0, "from_topic": "토픽명"}}]}}"""

# 실습: 해당 실습이 연습하는 개념
PRACTICE_USER = """각 실습 항목이 어떤 개념을 연습하는 실습인지 추출하세요.

규칙:
- 아래 '참고 개념 목록'에 해당 실습이 연습하는 개념이 있으면 그 이름을 그대로 사용하세요.
- 없으면 실습이 연습하는 개념명을 직접 작성하세요.
- "무엇을 하는 실습"이 아니라 "어떤 개념을 연습하는 실습"인지 중심으로 판단하세요.
- 개념명은 짧고 명확하게.

참고 개념 목록 (강제 아님):
{concepts}

실습 목록 ([인덱스] 이름):
{items}

아래 JSON 형식으로만 응답하세요:
{{"topics": [{{"index": 0, "from_topic": "개념명"}}]}}"""


def extract_topics(items, name_key, concept_names, is_practice=False):
    """예시/실습 항목의 토픽(예시) 또는 연습 개념(실습)을 추출."""
    if not items:
        return {}
    model = genai.GenerativeModel(MODEL)
    concepts_str = "\n".join(f"- {n}" for n in concept_names)
    items_str = "\n".join(f"[{i}] {it[name_key]}" for i, it in enumerate(items))
    template = PRACTICE_USER if is_practice else EXAMPLE_USER
    prompt = f"{EXTRACT_SYS}\n\n{template.format(concepts=concepts_str, items=items_str)}"
    resp = model.generate_content(prompt, generation_config=genai.GenerationConfig(
        response_mime_type="application/json", temperature=settings.llm_temperature))
    out = {}
    for a in _parse_json(resp.text).get("topics", []):
        try:
            idx = int(a["index"])
        except (KeyError, ValueError, TypeError):
            continue
        topic = a.get("from_topic", "").strip()
        if topic:
            out[idx] = topic
    return out

In [15]:
# ── 날짜별 from_topic 부여 → final json + xlsx 재기록 ─────────────────
def nearest_concept_name(item, concepts):
    """LLM 추출 실패 시 fallback: 위치상 가장 가까운 개념명."""
    before = [(i, c) for i, c in enumerate(concepts) if c["start"] <= item["start"]]
    pool = before or list(enumerate(concepts))
    i = min(pool, key=lambda ic: abs(item["start"] - ic[1]["start"]))[0]
    return concepts[i]["concept_name"]


def save_xlsx(path, rows):
    pd.DataFrame(rows).to_excel(path, index=False)
    wb = openpyxl.load_workbook(path); ws = wb.active
    for col, w in {"A": 12, "B": 32, "C": 12, "D": 40, "E": 45, "F": 65}.items():
        ws.column_dimensions[col].width = w
    for row in ws.iter_rows(min_row=2):
        for c in row:
            c.alignment = Alignment(wrap_text=True, vertical="top")
    wb.save(path)


for date in DATES:
    objs, items_by_kind = {}, {}
    for kind, key in KIND.items():
        objs[kind] = json.loads((GS_LABEL / f"{date}_{kind}_goldset_final.json").read_text(encoding="utf-8"))
        items_by_kind[kind] = objs[kind][key]

    concepts = items_by_kind["concept"]
    concept_names = [c["concept_name"] for c in concepts]

    # ── 개념: from_topic = 개념명 자체 ───────────────────────────────
    for c in concepts:
        c["from_topic"] = c["concept_name"]

    # ── 예시: 다루는 주제 추출 / 실습: 연습하는 개념 추출 ─────────────
    n_fallback = {"example": 0, "practice": 0}
    for kind, is_pr in (("example", False), ("practice", True)):
        items = items_by_kind[kind]
        extracted = extract_topics(items, NAME[kind], concept_names, is_practice=is_pr)
        for i, it in enumerate(items):
            if i in extracted:
                it["from_topic"] = extracted[i]
            else:
                it["from_topic"] = nearest_concept_name(it, concepts)
                n_fallback[kind] += 1

    # ── 저장 ─────────────────────────────────────────────────────────
    for kind, key in KIND.items():
        items = items_by_kind[kind]
        nk = NAME[kind]
        ordered = [{nk: it[nk], "start": it["start"], "end": it["end"],
                    "key_sentence": it.get("key_sentence", ""),
                    "timestamp": it.get("timestamp", ""),
                    "from_topic": it["from_topic"], "text": it.get("text", "")}
                   for it in items]
        objs[kind][key] = ordered
        (GS_LABEL / f"{date}_{kind}_goldset_final.json").write_text(
            json.dumps(objs[kind], ensure_ascii=False, indent=2), encoding="utf-8")
        rows = [{"timestamp": it["timestamp"], nk: it[nk], "span": f"{it['start']}~{it['end']}",
                 "from_topic": it["from_topic"], "key_sentence": it["key_sentence"], "text": it["text"]}
                for it in ordered]
        save_xlsx(GS_LABEL / f"{date}_{kind}_goldset_final.xlsx", rows)

    print(f"\n[{date}]  개념 {len(concepts)} / 예시 {len(items_by_kind['example'])}(fallback {n_fallback['example']}) / 실습 {len(items_by_kind['practice'])}(fallback {n_fallback['practice']})")
    print("  실습 from_topic:")
    for p in items_by_kind["practice"]:
        print(f"    [{p['start']}~{p['end']}] {p['practice_name'][:35]:35} → {p['from_topic'][:40]}")

print("\n완료.")


[2026-02-10]  개념 17 / 예시 15(fallback 0) / 실습 26(fallback 0)
  실습 from_topic:
    [131~218] 테이블 생성 및 INNER JOIN 실습              → 이너 조인 (INNER JOIN)
    [248~262] LEFT OUTER JOIN 사용하기                → 레프트 아우터 조인 (LEFT OUTER JOIN)
    [276~292] LEFT OUTER JOIN 테이블 순서 변경 실습        → 아우터 조인의 특징 (TFN 규칙)
    [319~326] RIGHT OUTER JOIN 구현                 → 라이트 아우터 조인 (RIGHT OUTER JOIN)
    [362~390] 5번: X, Y 테이블 모든 조합 결과 출력            → 크로스 조인 (CROSS JOIN)
    [415~478] Q1 테이블 생성 및 데이터 입력                  → 테이블 생성 및 데이터 입력 (DDL/DML)
    [489~519] EMP, DEPT 테이블 조인 문제 (1-8번)          → 조인 쿼리 종합 실습
    [521~570] INNER JOIN으로 사원 이름과 부서명 조회          → 이너 조인 (INNER JOIN)
    [582~608] LEFT OUTER JOIN으로 모든 사원과 부서명 조회     → 레프트 아우터 조인 (LEFT OUTER JOIN)
    [610~619] RIGHT OUTER JOIN으로 모든 부서와 사원 조회     → 라이트 아우터 조인 (RIGHT OUTER JOIN)
    [629~640] 4번. Full Outer Join 구현              → 풀 아우터 조인 (FULL OUTER JOIN)
    [659~706] 6번. Self Join으로 사원과 매니저 출력          → 셀프 조인 (SELF JOIN)
    [708~741] ANSI 